In [52]:
import os, io, time, glob, joblib
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_recall_fscore_support, roc_auc_score, average_precision_score,
                             accuracy_score, balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay)

BASE = "https://environment.data.gov.uk/hydrology"

# where - pick a river or a point on the map
RIVER_NAME = "River Exe"            # used if LAT is None
LAT, LONG, DIST_KM = None, None, 10
STATION_INDEX = 0                   # which station from the candidate list to use
RAIN_DIST_KM = 15                   # look for rain gauges this close to the river station
MAX_RAIN_STATIONS = 3

# when
START, END = "2022-10-01", "2025-10-01"

# Model settings
HORIZON_H  = 24 #this predicts the crossing within this many hours
HIGH_LEVEL_PCT = 0.95 # This basically is the percentile of the training period levels
TEST_FRACTION = 0.25 # final 25% of the timeline will be the test set

CACHE_DIR = "hydro_cache"
MODEL_PATH = "flood_rf.joblib"
SEED = 42
os.makedirs(CACHE_DIR, exist_ok=True)

In [53]:
#API Helpers

In [54]:
HEADERS = {"User-Agent": "ibmz-datathon-river-alert"}

def _get(url, params=None, retries=4 ,pause=1.0):
    for attempt in range(retries):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=180)
        except requests.exceptions.RequestException:
            if attempt == retries - 1:
                raise
            time.sleep(5 * (attempt + 1)); continue
        if r.status_code == 200:
            time.sleep(pause)
            return r
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(5 * (attempt + 1)); continue
        raise RuntimeError(f'API error {r.status_code} for {r.url}\n{r.text[:300]}')
    raise RuntimeError(f'Gave up after {retries} attempts: {url}')

def api_json(path, params=None):
    return _get(f'{BASE}{path}.json', params).json()

def api_csv(path, params=None):
    text = _get(f'{BASE}{path}.csv', params).text
    return pd.read_csv(io.StringIO(text)) if text.strip() else pd.DataFrame()

def last_segment(url):
    return url.strip("/").split("/")[-1]

In [55]:
p = {"observedProperty": "waterLevel", "status.label": "Active", "_limit": 50}
if LAT is not None:
    p.update({"lat": LAT, "long": LONG, "dist": DIST_KM})
else:
    p["riverName"] = RIVER_NAME
items = api_json("/id/stations", p)["items"]
assert items, "No stations found - try a different RIVER_NAME or set LAT/LONG."

cand = pd.DataFrame([{
    "idx": i, "label": s.get("label"), "river": s.get("riverName"), "town": s.get("town"),
    "guid": last_segment(s["@id"]), "lat": s.get("lat"), "long": s.get("long")}
    for i, s in enumerate(items)])
print(cand[["idx", "label", "river", "town"]].to_string(index=False))

def find_measure(guid, prop, period_name="15min"):
    ms = api_json("/id/measures", {"station": guid, "observedProperty": prop,
                                   "periodName": period_name, "observationType": "Qualified"})["items"]
    if not ms:
        return None
    ms.sort(key=lambda m: bool(m.get("qualifier")))   # prefer the plain series over qualified variants
    return last_segment(ms[0]["@id"])

level_measure, station = None, None
for i in range(STATION_INDEX, len(cand)):
    level_measure = find_measure(cand.guid[i], "waterLevel")
    if level_measure:
        station = cand.iloc[i]; break
assert level_measure, "None of the candidate stations has a 15-minute level series."
print("\nUsing station:", station.label, "|", station.river, "| measure:", level_measure)

rain_measures = []
rs = api_json("/id/stations", {"observedProperty": "rainfall", "status.label": "Active", "lat": float(station.lat),
                               "long": float(station.long), "dist": RAIN_DIST_KM, "_limit": 20})["items"]
for s in rs:
    m = find_measure(last_segment(s["@id"]), "rainfall")
    if m:
        rain_measures.append(m)
    if len(rain_measures) >= MAX_RAIN_STATIONS:
        break
print("Rain gauges found:", len(rain_measures))
if not rain_measures:
    print("WARNING: no rain gauges nearby - rainfall features will be zero. Increase RAIN_DIST_KM.")

 idx      label     river town
   0 Thorverton River Exe None
   1     Pixton River Exe None
   2 Stoodleigh River Exe None
   3     Exford River Exe None
   4 Trews Weir River Exe None

Using station: Thorverton | River Exe | measure: 3c4d4f78-2d0e-474a-b884-65a9daca18fb-level-i-900-m-qualified
Rain gauges found: 3


In [56]:
# Download readings

In [57]:
def year_chunks(start, end, days=365):
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    while s < e:
        n = min(s + pd.Timedelta(days=days), e)
        yield s.strftime("%Y-%m-%d"), n.strftime("%Y-%m-%d")
        s = n

def fetch_readings(measure, start, end, use_cache=True):
    parts = []
    for a, b in year_chunks(start, end):
        f = os.path.join(CACHE_DIR, f"{measure}_{a}_{b}.csv")
        df = None
        if use_cache and os.path.exists(f):
            try: df = pd.read_csv(f)
            except pd.errors.EmptyDataError: df = pd.DataFrame()
        if df is None:
            df = api_csv(f"/id/measures/{measure}/readings",
                         {"mineq-date": a, "max-date": b, "_view": "min", "_limit": 200000})
            if use_cache: df.to_csv(f, index=False)
        parts.append(df)
    df = pd.concat(parts, ignore_index=True)
    if df.empty or "value" not in df:
        return pd.Series(dtype=float)
    df["dateTime"] = pd.to_datetime(df["dateTime"], utc=True)
    return df.drop_duplicates("dateTime").set_index("dateTime")["value"].astype(float).sort_index()

def load_hourly(level_measure, rain_measures, start, end, use_cache=True):
    level = fetch_readings(level_measure, start, end, use_cache).resample("1h").max()   # hourly peak level
    rains = [fetch_readings(m, start, end, use_cache).resample("1h").sum(min_count=1) for m in rain_measures]
    rain = pd.concat(rains, axis=1).mean(axis=1) if rains else pd.Series(0.0, index=level.index)
    df = pd.DataFrame({"level": level, "rain": rain})
    df = df.reindex(pd.date_range(df.index.min(), df.index.max(), freq="1h"))
    df["level"] = df["level"].interpolate(limit=6)      # patch short sensor gaps only
    df["rain"] = df["rain"].fillna(0.0)                 # simplified it - missing rain counts as no rain
    return df

hourly = load_hourly(level_measure, rain_measures, START, END)
print(hourly.describe().round(2))
print("Hours with a valid level reading:", int(hourly.level.notna().sum()), "of", len(hourly))

          level      rain
count  26304.00  26304.00
mean       0.64      0.12
std        0.34      0.50
min        0.18      0.00
25%        0.41      0.00
50%        0.54      0.00
75%        0.79      0.00
max        2.91     24.07
Hours with a valid level reading: 26304 of 26304


In [58]:
def build_features(df):
    f = pd.DataFrame(index=df.index)
    f["level"] = df.level
    for h in (1, 3, 6, 12, 24):
        f[f"level_chg_{h}h"] = df.level.diff(h)
    for h in (6, 24, 72):
        f[f"level_max_{h}h"] = df.level.rolling(h).max()
    f["level_mean_72h"] = df.level.rolling(72).mean()
    for h in (1, 3, 6, 12, 24, 72, 168):
        f[f"rain_{h}h"] = df.rain.rolling(h, min_periods=1).sum()
    f["hour"] = df.index.hour
    f["month"] = df.index.month
    return f

def future_max(level, h):
    # max level over the NEXT h hours (t+1 ... t+h)
    return level.iloc[::-1].rolling(h, min_periods=h).max().iloc[::-1].shift(-1)

valid = hourly.dropna(subset=["level"])
split_time = valid.index[int(len(valid) * (1 - TEST_FRACTION))]
gap = pd.Timedelta(hours=HORIZON_H)

# High-level threshold comes from the TRAINING period only (no peeking at the test period)
level_threshold = hourly.level[hourly.index < split_time - gap].quantile(HIGH_LEVEL_PCT)
print(f"High-level threshold: {level_threshold:.3f} m   |   test period starts {split_time:%Y-%m-%d}")

feat = build_features(hourly)
fut = future_max(hourly.level, HORIZON_H)
data = feat.assign(target=(fut >= level_threshold).astype(int))
data = data[(hourly.level < level_threshold) & fut.notna()].dropna()

train = data[data.index < split_time - gap]       # the gap stops training windows overlapping the test period
test = data[data.index >= split_time]
FEATURES = [c for c in data.columns if c != "target"]
print(f"Train rows: {len(train)} ({train.target.mean():.1%} positive)  |  Test rows: {len(test)} ({test.target.mean():.1%} positive)")
assert train.target.sum() > 5 and test.target.sum() > 0, \
    "Too few flood events. Use a longer START/END, a lower HIGH_LEVEL_PCT, or a flashier river."

High-level threshold: 1.368 m   |   test period starts 2024-12-31
Train rows: 18644 (4.8% positive)  |  Test rows: 6460 (1.9% positive)


In [59]:
#training, choosing the alert threshold and evaluation

In [60]:
def new_rf():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight="balanced_subsample",
                                  n_jobs=-1, random_state=SEED)

# validation slice = last 20% of the training period (with a gap)
val_start = train.index[int(len(train) * 0.8)]
core, val = train[train.index < val_start - gap], train[train.index >= val_start]
rf = new_rf().fit(core[FEATURES], core.target)

alert_threshold = 0.5
if val.target.sum() > 0:
    pv = rf.predict_proba(val[FEATURES])[:, 1]
    grid = np.linspace(0.05, 0.95, 19)
    f1s = [precision_recall_fscore_support(val.target, pv >= t, average="binary", zero_division=0)[2] for t in grid]
    alert_threshold = float(grid[int(np.argmax(f1s))])
print(f"Alert threshold (probability): {alert_threshold:.2f}")

rf = new_rf().fit(train[FEATURES], train.target)      # final model on all training data
proba = rf.predict_proba(test[FEATURES])[:, 1]
pred = (proba >= alert_threshold).astype(int)

prec, rec, f1, _ = precision_recall_fscore_support(test.target, pred, average="binary", zero_division=0)
print(f"\nTest period results (future data the model never saw)")
print(f"  Accuracy                 : {accuracy_score(test.target, pred):.3f}")
print(f"  'Always no flood' accuracy: {1 - test.target.mean():.3f}   <- why accuracy alone is misleading")
print(f"  Balanced accuracy        : {balanced_accuracy_score(test.target, pred):.3f}")
print(f"  Precision (alerts that were right): {prec:.3f}")
print(f"  Recall (floods that were caught)  : {rec:.3f}")
print(f"  F1                       : {f1:.3f}")
print(f"  ROC-AUC                  : {roc_auc_score(test.target, proba):.3f}")
print(f"  Average precision (PR-AUC): {average_precision_score(test.target, proba):.3f}")

Alert threshold (probability): 0.30

Test period results (future data the model never saw)
  Accuracy                 : 0.975
  'Always no flood' accuracy: 0.981   <- why accuracy alone is misleading
  Balanced accuracy        : 0.636
  Precision (alerts that were right): 0.306
  Recall (floods that were caught)  : 0.283
  F1                       : 0.294
  ROC-AUC                  : 0.876
  Average precision (PR-AUC): 0.216


In [61]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))

ConfusionMatrixDisplay(confusion_matrix(test.target, pred), display_labels=["no flood", "flood <24h"]).plot(
    ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion matrix (test period)")

imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values().tail(12)
imp.plot.barh(ax=axes[1]); axes[1].set_title("What the model relies on")

lvl = hourly.level[hourly.index >= split_time]
axes[2].plot(lvl.index, lvl.values, lw=0.8, label="river level (m)")
axes[2].axhline(level_threshold, color="red", ls="--", label="high-level threshold")
al = test.index[pred == 1]
axes[2].scatter(al, hourly.level.reindex(al), s=10, color="orange", zorder=3, label="alert raised")
axes[2].set_title("Test period: level and alerts"); axes[2].legend(); axes[2].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

<Figure size 1800x480 with 3 Axes>

In [ ]:
# save the model

In [62]:
joblib.dump({"model": rf, "features": FEATURES, "alert_threshold": alert_threshold,
             "level_threshold": float(level_threshold), "horizon_h": HORIZON_H,
             "level_measure": level_measure, "rain_measures": rain_measures,
             "station": str(station.label)}, MODEL_PATH)
print("Saved", MODEL_PATH)

def predict_now(days=10):
    b = joblib.load(MODEL_PATH)
    now = pd.Timestamp.now(tz="UTC")
    start, end = (now - pd.Timedelta(days=days)).strftime("%Y-%m-%d"), (now + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    df = load_hourly(b["level_measure"], b["rain_measures"], start, end, use_cache=False)
    f = build_features(df).dropna()
    if f.empty:
        print("Not enough recent data to build features."); return None
    row = f.iloc[[-1]]
    p = float(b["model"].predict_proba(row[b["features"]])[0, 1])
    lvl, age = float(row["level"].iloc[0]), now - row.index[-1]

    if lvl >= b["level_threshold"]:   status = "ALREADY ABOVE high-level threshold"
    elif p >= b["alert_threshold"]:   status = "ALERT - level likely to cross the threshold"
    elif p >= b["alert_threshold"] / 2: status = "WATCH - elevated risk"
    else:                             status = "LOW risk"

    print(f"Station : {b['station']}")
    print(f"Latest reading: {lvl:.2f} m at {row.index[-1]:%Y-%m-%d %H:%M} UTC ({age.total_seconds()/3600:.1f} h old)")
    print(f"High-level threshold: {b['level_threshold']:.2f} m")
    print(f"P(cross within {b['horizon_h']}h): {p:.0%}   ->   {status}")
    return p

Saved flood_rf.joblib


In [ ]:
# makes a live api call and predicts the chances of flooding

In [63]:
predict_now()

Station : Thorverton
Latest reading: 0.35 m at 2026-10-18 01:00 UTC (0.5 h old)
High-level threshold: 1.37 m
P(cross within 24h): 0%   ->   LOW risk


0.0